# Buổi E7: Thực hành tổng hợp, ensemble có thật sự đáng dùng?

**Hiện trường.** Bưu điện (câu chuyện của E2) chốt bài toán: máy đọc **chữ số viết tay**; ca nào máy **không chắc** thì chuyển cho nhân viên đọc tay. Chị Hạnh cần chọn mô hình. Đội kỹ thuật ủng hộ "dùng ensemble cho chắc". Cô muốn **bằng chứng**: ensemble có hơn các mô hình đơn giản không, và đáng cái giá vận hành không?

Buổi này là **tổng ôn phần E** (và nhắc D) trên một dự án trọn vẹn:

1. đặt câu hỏi, thước đo và **mốc đơn giản**; niêm phong tập thi;
2. so **nhiều ứng viên** (đơn giản đến ensemble) bằng kiểm định chéo lặp, ghi cả **thời gian**;
3. chọn bằng **quy tắc một sai số chuẩn** có tính **chi phí vận hành**;
4. kiểm tra **ghép có đáng** không;
5. mở tập thi **một lần**; soi **lỗi theo từng chữ số**; đặt **ngưỡng chuyển cho người** theo độ chắc chắn;
6. viết **bản báo cáo**.

**Mindset của người làm dữ liệu:** ensemble là công cụ, không phải mục tiêu. Mô hình tốt là mô hình **đủ tốt, dễ vận hành, và ta biết nó sai ở đâu**.

**Lưu ý:** dữ liệu mẫu để học, không phải thư tín thật.

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 1 | Câu hỏi, thước đo, mốc ngu, niêm phong tập thi |
| Chặng 2 | Bảy ứng viên, đo công bằng và đo thời gian |
| Chặng 3 | Chọn: đơn giản nhất còn nằm trong một sai số chuẩn |
| Chặng 4 | Ghép có đáng không? |
| Chặng 5 | Mở tập thi một lần và ngưỡng chuyển cho người |
| Chặng 6 | Bản báo cáo |

## Chặng 1: Câu hỏi, thước đo, mốc ngu, niêm phong tập thi

**Chốt trước khi chạy:**
- **Thước đo chính: độ đúng** (tỉ lệ chữ số đọc đúng). Vì ca máy không chắc sẽ chuyển cho người, thước phụ là **độ đúng trên phần máy tự quyết** và **tỉ lệ phải chuyển cho người**.
- **Mốc ngu:** luôn đoán chữ số xuất hiện nhiều nhất; và **mốc đơn giản:** hồi quy logistic có chuẩn hóa.
- **Niêm phong** 25% làm tập thi, chỉ mở ở chặng 5.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=2026, stratify=y)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("train:", X_train.shape, "| sealed test:", X_test.shape, "| dummy accuracy:", round(dummy.score(X_train, y_train), 3))

train: (1347, 64) | sealed test: (450, 64) | dummy accuracy: 0.102


**Giải thích từng dòng**

- `train_test_split(..., random_state=2026, stratify=y)`: chia 75/25 giữ tỉ lệ mười chữ số (A4); `X_test, y_test` **niêm phong**.
- `DummyClassifier(strategy="most_frequent")`: mốc ngu (C2).

**Cách đọc.** Tập học **1347** ảnh, tập thi niêm phong **450** ảnh. Mốc ngu đạt **10,2%** (khoảng 1/10, như đoán mò); mô hình nào cũng phải vượt xa mức này.

**Mang về chặng 1:** chốt thước đo, mốc, niêm phong tập thi **trước** khi so mô hình.

## Chặng 2: Bảy ứng viên, đo công bằng và đo thời gian

Bảy ứng viên, từ đơn giản đến ensemble: **logistic**, **k-NN**, **SVM**, **rừng ngẫu nhiên**, **cây cực ngẫu nhiên** (`ExtraTrees`, họ hàng của rừng, chọn ngưỡng chia ngẫu nhiên), **gradient boosting bản histogram** và **bỏ phiếu mềm** của ba mô hình mạnh. Đo bằng kiểm định chéo lặp (5 phần × 2 lần) **chỉ trên tập học**, kèm thời gian huấn luyện.

In [3]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
svm = make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(C=10, random_state=0), ensemble=False))
forest = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
hgb = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, random_state=0)
candidates = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "svm": svm,
    "forest": forest,
    "extra trees": ExtraTreesClassifier(n_estimators=300, random_state=0, n_jobs=-1),
    "hist boosting": hgb,
    "soft vote (svm+forest+hgb)": VotingClassifier([("svm", svm), ("rf", forest), ("hgb", hgb)], voting="soft"),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)
rows = []
for name, m in candidates.items():
    t0 = time.time()
    s = cross_val_score(m, X_train, y_train, cv=cv)
    rows.append({"model": name, "accuracy": s.mean(), "se": s.std() / np.sqrt(len(s)), "seconds (all 10 fits)": time.time() - t0})
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))

                     model  accuracy     se  seconds (all 10 fits)
                  logistic    0.9666 0.0041                 0.3268
                       knn    0.9662 0.0036                 0.1331
                       svm    0.9770 0.0034                 2.6030
                    forest    0.9740 0.0021                 9.3046
               extra trees    0.9833 0.0026                 7.4437
             hist boosting    0.9685 0.0030                30.5324
soft vote (svm+forest+hgb)    0.9822 0.0022                42.7550


**Giải thích từng dòng**

- `ExtraTreesClassifier(n_estimators=300)`: rừng "cực ngẫu nhiên": giống rừng nhưng ngưỡng chia chọn ngẫu nhiên thay vì tối ưu; thường nhanh hơn.
- `VotingClassifier([(...), ...], voting="soft")`: bỏ phiếu mềm ba mô hình (E5).
- `s.std() / np.sqrt(len(s))`: sai số chuẩn của điểm trung bình (D7).
- `"seconds (all 10 fits)"`: tổng thời gian cho 10 lần học và chấm: để thấy **chi phí** của từng ứng viên.

**Cách đọc.**
- **Con số:** điểm tốt nhất **98,33%** (extra trees); thấp nhất **96,62%** (knn). Hồi quy logistic (mốc đơn giản): **96,66%**.
- **Bức tranh:** hầu hết các mô hình mạnh nằm gần nhau; ensemble **không tự động thắng**. Hãy đối chiếu chênh lệch với cột `se` và cột thời gian.
- **Câu hỏi tiếp:** chọn ai khi nhiều ứng viên sát nhau?

**Mang về chặng 2:** đo ứng viên công bằng, ghi cả sai số chuẩn **và** thời gian.

## Chặng 3: Chọn bằng một sai số chuẩn, có tính chi phí vận hành

**Quy tắc:** lấy ứng viên tốt nhất, tính ngưỡng = `điểm tốt nhất − sai số chuẩn của nó`; trong các ứng viên **nằm trên ngưỡng**, chọn cái **đơn giản nhất** theo thứ tự: logistic, k-NN, SVM, cây (rừng, cực ngẫu nhiên), boosting, bỏ phiếu. Ta xếp thứ tự này theo **độ khó vận hành**, một lựa chọn của người soạn có thể thảo luận.

In [4]:
order = ["logistic", "knn", "svm", "forest", "extra trees", "hist boosting", "soft vote (svm+forest+hgb)"]
best_row = res.loc[res["accuracy"].idxmax()]
floor = best_row["accuracy"] - best_row["se"]
within = res[res["accuracy"] >= floor].copy()
within["rank"] = within["model"].map(order.index)
chosen_name = within.sort_values("rank").iloc[0]["model"]
print("floor:", round(floor, 4))
print(within.sort_values("rank")[["model", "accuracy", "se"]].round(4).to_string(index=False))
print("chosen:", chosen_name)

floor: 0.9807
                     model  accuracy     se
               extra trees    0.9833 0.0026
soft vote (svm+forest+hgb)    0.9822 0.0022
chosen: extra trees


**Giải thích từng dòng**

- `res["accuracy"] >= floor`: lọc các ứng viên có điểm không thấp hơn ngưỡng: những người "ngang" người tốt nhất.
- `within["model"].map(order.index)`: đổi tên mô hình thành vị trí trong danh sách `order` (số nhỏ là đơn giản).
- `.sort_values("rank").iloc[0]`: lấy ứng viên đơn giản nhất trong nhóm ngang nhau.

**Cách đọc.**
- **Con số:** ngưỡng **98,07%**; có **2** ứng viên nằm trên ngưỡng; ta chọn **extra trees**.
- **Bức tranh:** quy tắc này **không hứa chọn mô hình điểm cao nhất**, mà chọn mô hình **đơn giản nhất không thua rõ**. Nếu bạn xếp thứ tự đơn giản khác (ví dụ ưu tiên dự đoán nhanh), kết quả có thể khác; hãy ghi rõ tiêu chí trong báo cáo.

**Mang về chặng 3:** chọn đơn giản nhất còn nằm trong một sai số chuẩn của người tốt nhất, và nói rõ thứ tự "đơn giản" của bạn.

## Chặng 4: Ghép có đáng không?

**Câu hỏi:** bỏ phiếu ba mô hình có hơn **mô hình riêng tốt nhất trong ba** một cách rõ ràng không?

So **cùng cách chia** (kiểm định chéo lặp có cố định hạt giống nên hai mô hình thấy cùng các phần): xét **hiệu điểm theo từng phần** và xem hiệu đó lớn hơn độ lệch của nó không (D1, chặng 4).

In [5]:
three = res[res["model"].isin(["svm", "forest", "hist boosting"])]
best_of_three = three.loc[three["accuracy"].idxmax(), "model"]
a_scores = cross_val_score(candidates[best_of_three], X_train, y_train, cv=cv)
v_scores = cross_val_score(candidates["soft vote (svm+forest+hgb)"], X_train, y_train, cv=cv)
diff = v_scores - a_scores
print("best of the three:", best_of_three, "| vote - best single (per fold) mean:", round(diff.mean() * 100, 2), "points | std:", round(diff.std() * 100, 2), "points | folds where vote wins:", int((diff > 0).sum()), "of", len(diff))

best of the three: svm | vote - best single (per fold) mean: 0.52 points | std: 0.58 points | folds where vote wins: 6 of 10


**Giải thích từng dòng**

- `res["model"].isin([...])`: lọc ba mô hình thành viên.
- `diff = v_scores - a_scores`: hiệu điểm theo từng phần: số dương nghĩa là bỏ phiếu thắng ở phần đó. Dùng được vì hai lần đo dùng **cùng cách chia** (`cv` cố định hạt giống).
- `(diff > 0).sum()`: đếm số phần bỏ phiếu thắng.

**Cách đọc.**
- **Con số:** bỏ phiếu hơn mô hình riêng tốt nhất (**svm**) trung bình **0,52** điểm phần trăm, độ lệch **0,58**, thắng **6/10** phần.
- **Bức tranh:** nếu chênh lệch trung bình **không lớn hơn nhiều** độ lệch, ghép **không có lợi thế rõ**; nếu có, hãy đặt vào cán cân với **chi phí gấp ba** (huấn luyện, dự đoán, bảo trì).

**Mang về chặng 4:** ghép đáng khi hiệu điểm lớn hơn hẳn độ lệch và chi phí chấp nhận được.

## Chặng 5: Mở tập thi một lần và ngưỡng chuyển cho người

Học lại mô hình đã chọn trên **toàn bộ tập học**, chấm **một lần** trên tập thi. Sau đó: ca nào mô hình **không chắc** (xác suất cao nhất thấp) thì **chuyển cho người**. Ta vẽ **độ đúng trên phần máy tự quyết** theo tỉ lệ chuyển cho người, và chọn **ngưỡng trên tập học** (bằng dự đoán ngoài lượt) chứ không trên tập thi.

In [6]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict
final = candidates[chosen_name].fit(X_train, y_train)
pred = final.predict(X_test)
acc = (pred == y_test).mean()
print("test accuracy (opened once):", round(acc, 4), "| errors:", int((pred != y_test).sum()), "of", len(y_test))
wrong_by_digit = pd.Series((pred != y_test).astype(int)).groupby(y_test).sum()
print("errors per true digit:", wrong_by_digit.to_dict())

test accuracy (opened once): 0.9822 | errors: 8 of 450
errors per true digit: {0: 1, 1: 0, 2: 0, 3: 2, 4: 0, 5: 1, 6: 0, 7: 0, 8: 3, 9: 1}


**Giải thích từng dòng**

- `candidates[chosen_name].fit(X_train, y_train)`: học lại mô hình đã chọn trên toàn bộ phần học.
- `pd.Series(...).groupby(y_test).sum()`: đếm số lỗi theo chữ số thật (`groupby` đã gặp ở A2): chữ số nào bị đọc sai nhiều nhất.

In [7]:
proba_oof = cross_val_predict(candidates[chosen_name], X_train, y_train, cv=5, method="predict_proba", n_jobs=-1)
conf_oof = proba_oof.max(axis=1)
ok_oof = proba_oof.argmax(axis=1) == y_train
conf_test = final.predict_proba(X_test).max(axis=1)
rows = []
for t in [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]:
    auto_oof = conf_oof >= t
    auto_te = conf_test >= t
    rows.append({"threshold": t, "to human (train OOF)": 1 - auto_oof.mean(), "auto accuracy (train OOF)": ok_oof[auto_oof].mean(),
                 "to human (test)": 1 - auto_te.mean(), "auto accuracy (test)": (pred[auto_te] == y_test[auto_te]).mean()})
handoff = pd.DataFrame(rows)
print(handoff.round(3).to_string(index=False))

 threshold  to human (train OOF)  auto accuracy (train OOF)  to human (test)  auto accuracy (test)
      0.00                 0.000                      0.981            0.000                 0.982
      0.50                 0.099                      0.999            0.093                 1.000
      0.70                 0.291                      1.000            0.296                 1.000
      0.80                 0.474                      1.000            0.436                 1.000
      0.90                 0.725                      1.000            0.693                 1.000
      0.95                 0.878                      1.000            0.864                 1.000


**Giải thích từng dòng**

- `cross_val_predict(..., method="predict_proba")`: xác suất ngoài lượt cho từng ảnh học (D5, E5): đo độ chắc **trung thực** mà không đụng tập thi.
- `proba.max(axis=1)`: độ chắc chắn của mô hình = xác suất cao nhất trong mười chữ số.
- `conf_oof >= t`: ca nào đạt ngưỡng thì máy tự quyết.
- `(1 - auto.mean())`: tỉ lệ ca phải chuyển cho người.

**Cách đọc.**
- **Con số:** độ đúng trên tập thi **98,22%** (8 lỗi trên 450 ảnh). Mô hình đã chọn: **extra trees**.
- **Ngưỡng:** đặt ngưỡng **0,9** thì (trên tập học, dự đoán ngoài lượt) phải chuyển **72,5%** ca cho người và độ đúng phần máy tự quyết **100,0%**; trên tập thi: chuyển **69,3%**, tự quyết đúng **100,0%**.
- **Bức tranh:** thêm ngưỡng đổi **công của người** lấy **độ đúng của máy**; hai cột trên tập học và tập thi gần nhau là dấu hiệu độ chắc chắn đáng tin (xem D5 nếu không). Chọn ngưỡng theo **giá của một ca đọc sai** so với **giá của một ca chuyển cho người**, chứ không theo con số đẹp.
- **Nói thật:** tập thi chỉ 450 ảnh nên mỗi lỗi là 0,22 điểm phần trăm; chênh lệch dưới 1 điểm giữa các hàng của bảng chưa chắc có thật.

**Mang về chặng 5:** mở tập thi một lần; đặt ngưỡng chuyển cho người trên dự đoán ngoài lượt của tập học.

## Chặng 6: Bản báo cáo gửi hội đồng bưu điện

**Câu hỏi:** máy đọc chữ số có dùng được, dùng thế nào?

**Quy trình đã làm:** chốt thước đo và mốc ngu; niêm phong tập thi; so bảy ứng viên bằng kiểm định chéo lặp kèm thời gian; chọn bằng quy tắc một sai số chuẩn; kiểm tra ghép; mở tập thi một lần; đặt ngưỡng chuyển cho người trên dự đoán ngoài lượt.

**Kết quả** (số cụ thể xem các chặng 2, 3, 5): mô hình chọn là mô hình **đơn giản nhất không thua rõ** người tốt nhất; ensemble được thử và so với cái giá của nó.

**Hạn chế:**
- ảnh là chữ số 8×8 điểm ảnh của bộ dữ liệu mẫu, **ảnh phong bì thật** sẽ nhiễu hơn, khác kiểu viết, khác độ phân giải;
- tập thi chỉ vài trăm ảnh nên khoảng dao động rộng;
- ta chưa kiểm tra theo nhóm người viết (D6): người viết mới có thể khó hơn.

**Khuyến nghị:**
1. dùng mô hình đã chọn làm **mô hình chính**; **đừng** triển khai ensemble nếu lợi ích nằm trong độ dao động;
2. đặt ngưỡng chuyển cho người theo **chi phí thật** (giá một ca sai, giá một ca đọc tay);
3. thử lại trên **ảnh bưu điện thật** trước khi triển khai và theo dõi định kỳ.

**Bài học nghề:** *"dùng ensemble cho chắc"* không phải là lập luận; **bằng chứng** (kiểm định chéo, tập thi một lần, chi phí) mới là lập luận.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 1 | Chốt thước đo, mốc ngu, niêm phong tập thi trước khi chạy. |
| 2 | So ứng viên từ đơn giản đến ensemble, ghi sai số chuẩn và thời gian. |
| 3 | Chọn đơn giản nhất còn trong một sai số chuẩn; nói rõ thứ tự đơn giản. |
| 4 | Ghép đáng khi hiệu điểm lớn hơn hẳn độ lệch và chi phí chấp nhận được. |
| 5 | Mở tập thi một lần; ngưỡng chuyển cho người đặt trên dự đoán ngoài lượt. |
| 6 | Báo cáo: kết quả, hạn chế, khuyến nghị dựa trên bằng chứng. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Đo công bằng nhiều lần | kiểm định chéo lặp | `RepeatedStratifiedKFold` |
| Rừng với ngưỡng chia ngẫu nhiên | cây cực ngẫu nhiên | `ExtraTreesClassifier` |
| Gradient boosting nhanh | boosting histogram | `HistGradientBoostingClassifier` |
| Hội đồng bỏ phiếu mềm | voting | `VotingClassifier(voting='soft')` |
| Ngưỡng chọn bằng điểm trừ sai số chuẩn | quy tắc một sai số chuẩn | `accuracy >= best - se` |
| Hiệu điểm theo từng phần | so sánh cặp đôi | `v_scores - a_scores` |
| Máy không chắc thì nhờ người | chuyển cho người theo độ chắc | `predict_proba(...).max(axis=1)` |

**Thử thách tự làm**

1. Đổi thứ tự `order` trong chặng 3: ưu tiên mô hình dự đoán nhanh nhất. Mô hình được chọn có đổi không? Điều đó nói gì về việc "đơn giản" cần được định nghĩa?
2. Thêm `class_weight='balanced'` hoặc một ứng viên mới (ví dụ `GradientBoostingClassifier` của E3) vào chặng 2. Nó có nằm trên ngưỡng không, và thời gian ra sao?
3. Tính chi phí: một ca đọc sai tốn 10 đơn vị, một ca chuyển cho người tốn 1 đơn vị. Ngưỡng nào ở chặng 5 cho tổng chi phí thấp nhất trên **tập học (ngoài lượt)**?
4. Dùng độ quan trọng hoán vị (E6) trên tập thi cho mô hình chọn: vùng điểm ảnh nào quan trọng nhất? Có giống bản đồ nhiệt ở E2 không?

**Checklist năng lực** (tự tick khi làm được)

- [ ] Chốt thước đo và mốc đơn giản trước khi so mô hình.
- [ ] So ứng viên kèm sai số chuẩn và thời gian.
- [ ] Áp dụng quy tắc một sai số chuẩn có tiêu chí đơn giản rõ ràng.
- [ ] Đánh giá ghép bằng hiệu điểm theo từng phần.
- [ ] Đặt ngưỡng chuyển cho người mà không đụng tập thi.
- [ ] Viết báo cáo có hạn chế và khuyến nghị dựa trên bằng chứng.

## Phần thưởng (Tip pro): Khi nào ensemble thật sự đáng


Ensemble thường **đáng** khi: (1) dữ liệu dạng **bảng** với quan hệ phức tạp (nhiều tương tác giữa cột), nơi cây và boosting vượt hẳn mô hình tuyến tính; (2) cần **từng phần trăm** vì sai sót đắt; (3) chi phí dự đoán và bảo trì chấp nhận được. Thường **không đáng** khi: (1) dữ liệu ít, mô hình đơn giản đã chạm trần; (2) cần **giải thích** từng quyết định cho khách hàng hoặc cơ quan quản lý; (3) mô hình phải chạy **trong vài mili giây** trên thiết bị nhỏ. Với ảnh, âm thanh, văn bản, **mạng nơ-ron** (phần G) mới là họ mô hình ưu thế; boosting và rừng thống trị ở bảng số liệu.